In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable
    
bronze_table = "agriculture_dev.bronze.agriculture_raw"
silver_table = "agriculture_dev.silver.agriculture_clean"
quarantine_table = "agriculture_dev.silver.agriculture_quarantine"

bronze_df = spark.table(bronze_table)
# Process all batches; latest accepted observation per farm becomes current Silver state.

silver_df = bronze_df.select(
    trim("Farm_ID").alias("farm_id"),
    initcap(trim(("Crop_Type"))).alias("crop_type"),
    col("Farm_Area_acres").cast("double").alias("farm_area_acres"),
    initcap(trim("Irrigation_Type")).alias("irrigation_type"),
    col("Fertilizer_Used_tons").cast("double").alias("fertilizer_used_tons"),
    col("Pesticide_Used_kg").cast("double").alias("pesticide_used_kg"),
    col("Yield_tons").cast("double").alias("yield_tons"),
    initcap(trim("Soil_Type")).alias("soil_type"),
    initcap(trim("Season")).alias("season"),
    col("Water_Usage_cubic_meters").cast("double").alias("water_usage_cubicmeters"),
    col("row_hash"),
    col("Src_name").alias("source_name"),
    col("Ingestion_timestamp").alias("ingestion_timestamp"),
    col("batch_id")
)

display(silver_df)
silver_df.printSchema()

validated_df = silver_df.withColumn(
    "validation_errors",
    concat_ws(
        ", ",
        when(col("farm_id").isNull() | (trim(col("farm_id")) == ""),"Missing Farm_ID"),
        when(col("crop_type").isNull() | (trim(col("crop_type")) == ""),"Missing Crop_Type"),
        when(col("farm_area_acres").isNull() | (col("farm_area_acres") < 0), "Invalid Farm_Area or less farm area"),
        when(col("fertilizer_used_tons").isNull() |(col("fertilizer_used_tons") < 0),"Invalid Fertilizer_Used"),
        when(col("pesticide_used_kg").isNull() |(col("pesticide_used_kg") < 0), "Invalid Pesticide_Used"),
        when(col("yield_tons").isNull() |(col("yield_tons") < 0),"Invalid Yield"),
        when(col("water_usage_cubicmeters").isNull() |(col("water_usage_cubicmeters") < 0),"Invalid Water_Usage"),
        when(col("irrigation_type").isNull() | (trim(col("irrigation_type")) == ""),"Missing Irrigation_Type"),
        when(col("soil_type").isNull() | (trim(col("soil_type")) == ""),"Missing Soil_Type"),
        when(col("season").isNull() |(trim(col("season")) == ""), "Missing Season" )))

display(validated_df)

valid_df = validated_df.filter(col("validation_errors") == "")

quarantine_df = validated_df.filter(col("validation_errors") != "")

print("Total records:", validated_df.count())
print("Valid records:", valid_df.count())
print("Quarantined records:", quarantine_df.count())


# Composite business key. Rule 3: same Farm_ID with a different crop/season/irrigation
# is a different business key and is naturally retained as a separate record.
dedup_key = ["farm_id", "crop_type", "season", "irrigation_type"]
measure_cols = ["farm_area_acres", "fertilizer_used_tons", "pesticide_used_kg", "yield_tons", "water_usage_cubicmeters"]

# NULL-safe signature of the measure columns so NULL and blank are compared explicitly
measure_sig = concat_ws("||", *[coalesce(col(c).cast("string"), lit("<NULL>")) for c in measure_cols])

key_stats = (valid_df.withColumn("measure_signature", measure_sig).groupBy(*dedup_key)
             .agg(count("*").alias("rows_in_group"),
                  count_distinct("measure_signature").alias("distinct_measures")))

# Rule 2 - same key, different measures: conflicting records, quarantine every member
conflict_keys = key_stats.filter(col("distinct_measures") > 1).select(*dedup_key)

# Rule 1 - same key, identical measures: exact duplicates, retain one per key
duplicate_keys = key_stats.filter((col("distinct_measures") == 1) & (col("rows_in_group") > 1)).select(*dedup_key)

# Rank within each key group: newest observation wins
w = Window.partitionBy(*dedup_key).orderBy(col("row_hash").asc(), col("batch_id").asc())
ranked = valid_df.withColumn("rn", row_number().over(w))

conflicting_records = (ranked.join(conflict_keys, on=dedup_key, how="inner")
                       .withColumn("validation_errors", lit("conflicting measures for the same business key")))

dup_group_records = ranked.join(duplicate_keys, on=dedup_key, how="inner")
retained_duplicates = dup_group_records.filter(col("rn") == 1).drop("rn")
dropped_duplicates = dup_group_records.filter(col("rn") > 1).drop("rn")

# Keys with no conflicts and no duplication pass straight through
unique_valid_df = (ranked.join(conflict_keys, on=dedup_key, how="left_anti")
                   .join(duplicate_keys, on=dedup_key, how="left_anti")
                   .drop("rn"))

# Rows that proceed to the Silver MERGE: unique keys + one retained row per exact-duplicate key
silver_ready_df = unique_valid_df.unionByName(retained_duplicates)

print("Valid records entering dedup:", valid_df.count())
print("Exact-duplicate groups (one retained each):", duplicate_keys.count())
print("Exact duplicate copies dropped:", dropped_duplicates.count())
print("Conflicting records quarantined:", conflicting_records.count())
print("Records proceeding to Silver MERGE:", silver_ready_df.count())


quarantine_df.write \
    .format("delta") \
    .mode("append") \
    .option("overwriteSchema", "true") \
    .saveAsTable(quarantine_table)

## apply merge
# Create the empty Silver target on the first run so the MERGE has a table to merge into
if not spark.catalog.tableExists(silver_table):
    silver_ready_df.write.format("delta").saveAsTable(silver_table)

target = DeltaTable.forName(spark, silver_table)

merge_condition = """
    target.farm_id = source.farm_id
    AND target.crop_type = source.crop_type
    AND target.season = source.season
    AND target.irrigation_type = source.irrigation_type
"""

(
    target.alias("target")
    .merge(
        silver_ready_df.alias("source"),
        merge_condition
    )
    .whenMatchedUpdateAll(
        condition="target.row_hash <> source.row_hash"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("Silver MERGE completed successfully")

farm_id,crop_type,farm_area_acres,irrigation_type,fertilizer_used_tons,pesticide_used_kg,yield_tons,soil_type,season,water_usage_cubicmeters,row_hash,source_name,ingestion_timestamp,batch_id
F001,Cotton,329.4,Sprinkler,8.14,2.21,14.44,Loamy,Kharif,76648.2,fb08cbe7ef4b4ea993b059fbc474f2011af049a1235f80fab219ac22937130c0,Agriculture System,2026-11-06T09:16:20.878Z,29
F002,Carrot,18.67,Manual,4.77,4.36,42.91,Peaty,Kharif,68725.54,9506c595083df02b6de5c10aaee02b959cd3faa07eb15555859c652ac0d5d30f,Agriculture System,2026-11-06T09:16:20.878Z,29
F003,Sugarcane,306.03,Flood,2.91,0.56,33.44,Silty,Kharif,75538.56,f5aa7fb388d6b02407f7cec1219389e877eab35b02c0b45e871fda4ebf76dfd3,Agriculture System,2026-11-06T09:16:20.878Z,29
F004,Tomato,380.21,Rain-fed,3.32,4.35,34.08,Silty,Zaid,45401.23,77b98a6bc4d18e43a306853db887a68d1e5f5b70090b25607391e2d91b17c417,Agriculture System,2026-11-06T09:16:20.878Z,29
F005,Tomato,135.56,Sprinkler,8.33,4.48,43.28,Clay,Zaid,93718.69,e5f24bc6d3d780b07b94a897261e7591b2ea77e7097a495a441ac8ccf86412d2,Agriculture System,2026-11-06T09:16:20.878Z,29
F006,Sugarcane,12.5,Sprinkler,6.42,2.25,38.18,Loamy,Zaid,46487.98,5787eaf1fe298c4c96bebd02a0997042d88df0afd95cedb43d48d8c59e80dc73,Agriculture System,2026-11-06T09:16:20.878Z,29
F007,Soybean,360.06,Drip,1.83,2.37,44.93,Sandy,Rabi,40583.57,7e52176490c74ec4cce2b346867148f2c558e1e911d39f9cd489a569336580b1,Agriculture System,2026-11-06T09:16:20.878Z,29
F008,Rice,464.6,Drip,5.18,0.91,4.23,Silty,Kharif,9392.38,724b54cae380496fd977f272b09f2a46dfdde738ded057531a0193ca8c12ffc8,Agriculture System,2026-11-06T09:16:20.878Z,29
F009,Maize,389.37,Drip,0.57,4.93,3.86,Peaty,Rabi,60202.14,ef18a3b0a261e160da92cfa3053ba1d0d97eaf34e96aabaa02741f0c55e4becf,Agriculture System,2026-11-06T09:16:20.878Z,29
F010,Soybean,184.37,Drip,2.18,2.67,17.25,Sandy,Kharif,90922.15,e48cded42c55b875359ba287393e7346b75029594a19c1b3ba88e6d7303b83a6,Agriculture System,2026-11-06T09:16:20.878Z,29


root
 |-- farm_id: string (nullable = true)
 |-- crop_type: string (nullable = true)
 |-- farm_area_acres: double (nullable = true)
 |-- irrigation_type: string (nullable = true)
 |-- fertilizer_used_tons: double (nullable = true)
 |-- pesticide_used_kg: double (nullable = true)
 |-- yield_tons: double (nullable = true)
 |-- soil_type: string (nullable = true)
 |-- season: string (nullable = true)
 |-- water_usage_cubicmeters: double (nullable = true)
 |-- row_hash: string (nullable = true)
 |-- source_name: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- batch_id: integer (nullable = true)



farm_id,crop_type,farm_area_acres,irrigation_type,fertilizer_used_tons,pesticide_used_kg,yield_tons,soil_type,season,water_usage_cubicmeters,row_hash,source_name,ingestion_timestamp,batch_id,validation_errors
F001,Cotton,329.4,Sprinkler,8.14,2.21,14.44,Loamy,Kharif,76648.2,fb08cbe7ef4b4ea993b059fbc474f2011af049a1235f80fab219ac22937130c0,Agriculture System,2026-11-06T09:16:20.878Z,29,
F002,Carrot,18.67,Manual,4.77,4.36,42.91,Peaty,Kharif,68725.54,9506c595083df02b6de5c10aaee02b959cd3faa07eb15555859c652ac0d5d30f,Agriculture System,2026-11-06T09:16:20.878Z,29,
F003,Sugarcane,306.03,Flood,2.91,0.56,33.44,Silty,Kharif,75538.56,f5aa7fb388d6b02407f7cec1219389e877eab35b02c0b45e871fda4ebf76dfd3,Agriculture System,2026-11-06T09:16:20.878Z,29,
F004,Tomato,380.21,Rain-fed,3.32,4.35,34.08,Silty,Zaid,45401.23,77b98a6bc4d18e43a306853db887a68d1e5f5b70090b25607391e2d91b17c417,Agriculture System,2026-11-06T09:16:20.878Z,29,
F005,Tomato,135.56,Sprinkler,8.33,4.48,43.28,Clay,Zaid,93718.69,e5f24bc6d3d780b07b94a897261e7591b2ea77e7097a495a441ac8ccf86412d2,Agriculture System,2026-11-06T09:16:20.878Z,29,
F006,Sugarcane,12.5,Sprinkler,6.42,2.25,38.18,Loamy,Zaid,46487.98,5787eaf1fe298c4c96bebd02a0997042d88df0afd95cedb43d48d8c59e80dc73,Agriculture System,2026-11-06T09:16:20.878Z,29,
F007,Soybean,360.06,Drip,1.83,2.37,44.93,Sandy,Rabi,40583.57,7e52176490c74ec4cce2b346867148f2c558e1e911d39f9cd489a569336580b1,Agriculture System,2026-11-06T09:16:20.878Z,29,
F008,Rice,464.6,Drip,5.18,0.91,4.23,Silty,Kharif,9392.38,724b54cae380496fd977f272b09f2a46dfdde738ded057531a0193ca8c12ffc8,Agriculture System,2026-11-06T09:16:20.878Z,29,
F009,Maize,389.37,Drip,0.57,4.93,3.86,Peaty,Rabi,60202.14,ef18a3b0a261e160da92cfa3053ba1d0d97eaf34e96aabaa02741f0c55e4becf,Agriculture System,2026-11-06T09:16:20.878Z,29,
F010,Soybean,184.37,Drip,2.18,2.67,17.25,Sandy,Kharif,90922.15,e48cded42c55b875359ba287393e7346b75029594a19c1b3ba88e6d7303b83a6,Agriculture System,2026-11-06T09:16:20.878Z,29,


Total records: 55
Valid records: 55
Quarantined records: 0
Valid records entering dedup: 55
Exact-duplicate groups (one retained each): 0
Exact duplicate copies dropped: 0
Conflicting records quarantined: 0
Records proceeding to Silver MERGE: 55
Silver MERGE completed successfully
